# R2-07 · Evaluación y validación — Profundización (opcional) 🔬

**Formación Pública — Línea B · Data Scientist · Notebook de profundización**

Este cuaderno es **opcional**. Si ya hiciste la lección de R2-07, aquí vamos al *porqué* y a las
herramientas finas: la **curva ROC/AUC** y el **umbral** que eliges tú, y la **calibración** de
probabilidades con `CalibratedClassifierCV`. Mismos datos reales de compras públicas.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, recall_score, brier_score_loss
import os, urllib.request
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)
df = pd.read_csv(CSV)
y = (df["tamano_proveedor"] == "Grande").astype(int)
CAT = ["categoria", "region_comprador"]
X = df[["cantidad", "monto_total"] + CAT]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
proba = modelo.predict_proba(Xte)[:, 1]
print("Listo: modelo entrenado, proba calculada.")

## 1. El umbral lo eliges tú: ROC y AUC

El modelo da una **probabilidad**; eres tú quien fija el **umbral** que la convierte en decisión. El **AUC** resume qué tan bien separa las clases en todos los umbrales.

### ✍️ Ejercicio 1 — Calcula el AUC del modelo

In [ ]:
auc = roc_auc_score(yte, proba)
print("AUC:", round(auc, 3))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert 0.5 < auc <= 1.0, "Un modelo util separa mejor que el azar (AUC > 0.5)"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'roc_auc_score(yte, proba).')
    print("   Detalle:", e)

## 2. Mover el umbral para subir el recall

Si te importa **no perderte** proveedores grandes (recall), baja el umbral. Tiene un costo (más falsos positivos), pero a veces el problema público lo justifica.

### ✍️ Ejercicio 2 — Encuentra un umbral que logre recall ≥ 0.5

In [ ]:
thr = 0.5
for t in np.linspace(0.5, 0.01, 50):
    if recall_score(yte, (proba >= t).astype(int)) >= 0.5:
        thr = t
        break
rec_thr = recall_score(yte, (proba >= thr).astype(int))
print("Umbral:", round(thr, 3), "| recall:", round(rec_thr, 3))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert rec_thr >= 0.5, "Bajando el umbral deberias alcanzar recall >= 0.5"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '(proba >= t).astype(int) son las predicciones con umbral t; mide su recall_score.')
    print("   Detalle:", e)

## 3. Calibrar las probabilidades

`CalibratedClassifierCV` ajusta las probabilidades para que sean más fieles. Compara el Brier.

### ✍️ Ejercicio 3 — Calibra el modelo y mide el Brier resultante

In [ ]:
cal = CalibratedClassifierCV(
    Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]),
    cv=3, method="sigmoid").fit(Xtr, ytr)
brier_cal = brier_score_loss(yte, cal.predict_proba(Xte)[:, 1])
print("Brier calibrado:", round(brier_cal, 4))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert 0.0 <= brier_cal <= 1.0
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'cal.predict_proba(Xte)[:, 1] son las probabilidades calibradas; pasalas a brier_score_loss.')
    print("   Detalle:", e)

---

*Fin de la profundización de R2-07.*